# Day 4, Notebook 1: what is typical, and what one record does to it

Yesterday you decided which records were usable and wrote down why. Today somebody asks what those records say.

That question has three honest answers and they disagree with each other. This notebook is about picking the one that does not mislead the person who asked.

Everything here runs on your cleaned file and nothing else. Describing data you have not cleaned is the mistake this week already made on purpose.

## Setup

One cell, at the top, so this notebook runs cold in a fresh Codespace.

`statistics` is part of the standard library and ships with Python. It is here so you can check your own arithmetic against something you did not write. You will compute the median by hand first, then let the library confirm it.

In [ ]:
import csv
import statistics

CLEANED_CSV = "C2_W01_D04_data_cleaned_STUDENT.csv"

with open(CLEANED_CSV) as f:
    records = list(csv.DictReader(f))

# Everything read from a CSV is text. Tuesday's rule, still true.
for r in records:
    r["amount"] = int(r["amount"])

amounts = [r["amount"] for r in records]

print("records loaded:", len(records))
print("first record:  ", records[0])

Forty-seven records. That is what yesterday's cleaning pass left after fifty went in and three came out.

If your count says something other than 47, stop here and check which file you opened. Every number below depends on this one.

## Where this is going, before we build any of it

Two statements about the file you just loaded. Both are true.

In [ ]:
mean_amount = sum(amounts) / len(amounts)
at_or_above = [a for a in amounts if a >= mean_amount]

print("The average order is Rs {:.2f}.".format(mean_amount))
print("Orders at or above that average:", len(at_or_above), "out of", len(amounts))

An average that describes one record out of forty-seven.

Nothing crashed. The arithmetic is correct. If you sent the first line to somebody who trusted you, they would plan around a number that fits nobody.

That gap between a correct number and an honest description is the whole of today.

## Section 1: three answers to "what is typical"

Before any code, do this on paper. Seven values, which are the amounts of the first seven records in your file.

```
2000   3000   4000   4500   5000   6500   10000
```

Three ways to answer "what does a typical one look like":

```
  MEAN     add them all, share the total out equally
             |
  MEDIAN   stand them in a line, walk to the middle, read that one
             |
  MODE     which exact value shows up most often
```

Work out the mean and the median with a pen. Then run the next cell.

In [ ]:
seven = amounts[:7]
print("the seven:", seven)
print("sum:      ", sum(seven))
print("mean:     ", sum(seven) / len(seven))
print("sorted:   ", sorted(seven))
print("median:   ", statistics.median(seven))

Mean Rs 5,000, median Rs 4,500. Close together, and either one would describe this set of seven honestly.

The mean used all seven values. The median used their order and one value in the middle. That difference does nothing here, which is exactly why the next cell matters.

### One record changes, and the two answers part company

Take the same seven values and make the last one Rs 80,000. Change nothing else.

In [ ]:
seven_with_a_big_one = sorted(seven)[:-1] + [80000]

print("original: ", sorted(seven))
print("  mean:   ", sum(seven) / len(seven))
print("  median: ", statistics.median(seven))
print()
print("changed:  ", seven_with_a_big_one)
print("  mean:   ", sum(seven_with_a_big_one) / len(seven_with_a_big_one))
print("  median: ", statistics.median(seven_with_a_big_one))

The mean tripled. The median did not move by one rupee.

That is not a quirk of these seven numbers. It is what the two statistics are built to do:

- The **mean** gives every record a vote, and the vote is weighted by size. A large record shouts.
- The **median** gives every record a vote of equal weight. A large record is just one more record standing to the right.

Choosing between them is choosing whether the loudest record gets to speak for the quiet ones.

### Mode, and why it earns nothing here

In [ ]:
counts_by_amount = {}
for a in amounts:
    counts_by_amount[a] = counts_by_amount.get(a, 0) + 1

most_common = sorted(counts_by_amount.items(), key=lambda pair: pair[1], reverse=True)[:3]
print("the three most repeated amounts:", most_common)
print("total records:", len(amounts))

The most common amount is Rs 4,500 and it appears three times out of forty-seven.

Mode answers "which exact value repeats", and on a money column almost nothing repeats. Ask it about a category and it becomes useful immediately: "the most common outcome" or "the most common segment" are real answers a stakeholder can use.

The rule worth keeping: mode is for categories, and a money column is not a category.

## Section 2: the whale

Run the mean on the real column again, this time with the two lines that expose it.

In [ ]:
mean_amount = sum(amounts) / len(amounts)
above = [a for a in amounts if a >= mean_amount]
below = [a for a in amounts if a < mean_amount]

print("mean amount over {} records: Rs {:.2f}".format(len(amounts), mean_amount))
print("records at or above Rs {:.2f}: {}".format(mean_amount, len(above)))
print("records below Rs {:.2f}: {}".format(mean_amount, len(below)))

### The deliberate failure of this half, and it never raises

```
mean amount over 47 records: Rs 18000.00
records at or above Rs 18000.00: 1
records below Rs 18000.00: 46
```

No traceback. No red text. Nothing in Tuesday's toolkit fires, because nothing went wrong in the sense Python understands.

The failure is that a true sentence produced a false impression. `try` and `except` cannot catch that. The only thing that catches it is looking at the shape before you speak.

Sort the column and the cause walks out on its own.

In [ ]:
print("the seven largest amounts in the file:")
for a in sorted(amounts)[-7:]:
    print("   Rs {:,}".format(a))

The second largest order in the file is Rs 17,400. The largest is Rs 480,000, which is twenty-seven times the one below it.

One record. Yesterday's cleaning pass looked at it and kept it, and kept it for the right reason: it is not a typo, not a text value and not a duplicate. It is a real order somebody really placed.

**A record can be perfectly correct and still wreck every summary it touches.**

In [ ]:
whale = max(amounts)
ordinary = [a for a in amounts if a != whale]

print("with the whale:    n={:2d}  mean=Rs {:>10,.2f}  median=Rs {:>9,.2f}".format(
    len(amounts), sum(amounts) / len(amounts), statistics.median(amounts)))
print("without the whale: n={:2d}  mean=Rs {:>10,.2f}  median=Rs {:>9,.2f}".format(
    len(ordinary), sum(ordinary) / len(ordinary), statistics.median(ordinary)))

Removing one record out of forty-seven cuts the mean by more than half. The median moves by Rs 200.

Now the question that decides whether you are an analyst or a decorator: **do you delete the whale?**

No. Yesterday you wrote a decisions log precisely so nobody could quietly remove an inconvenient record. The record stays in the file and the statistic changes. Deleting real data to make a number look tidier is how a report turns into fiction, and it is the kind of thing that surfaces six months later in front of people who did not do it.

### Milestone 1: what you can now answer

**Where this shows up in production.** National statistical agencies report *median* household income rather than the mean, because a small number of very high incomes drag the mean away from anything a household would recognise. The same convention runs through salary bands, insurance claim sizes, invoice amounts and basket totals. Any team that reports "average order value" on a marketplace without checking the tail first has shipped this bug.

**Interview question this section just made answerable.**

> A stakeholder asks for the average order value. One enormous order sits in the data. What number do you give them, and what do you say?

A complete answer has three parts. Give the median and name it as the median. State the count it rests on. Say the large order exists, that it is real and retained, and that quoting the mean would describe one record out of forty-seven. Volunteering the third part is what separates a good answer from a correct one.

## Section 3: spread, and a rule that runs without you

Typical is one number. Spread is how far the file wanders from it.

In [ ]:
print("min:   Rs {:>9,}".format(min(amounts)))
print("max:   Rs {:>9,}".format(max(amounts)))
print("range: Rs {:>9,}".format(max(amounts) - min(amounts)))

Range is one subtraction and the least stable number you will produce today. It is built from exactly two records out of forty-seven, and one of them is the whale. A statistic computed from two records tells you about those two records.

Yesterday you flagged an outlier by sorting and eyeballing the tail. Here is the same idea with a rule attached, so it runs on a file nobody has looked at.

```
   Q1                    Q3
    |                     |
 ---+---------------------+-------------------------|
    |<------ IQR -------->|                    upper fence
                                            Q3 + 1.5 x IQR
```

In [ ]:
quarters = statistics.quantiles(amounts, n=4)
q1, q3 = quarters[0], quarters[2]
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print("Q1:          Rs {:>9,.2f}".format(q1))
print("Q3:          Rs {:>9,.2f}".format(q3))
print("IQR:         Rs {:>9,.2f}".format(iqr))
print("upper fence: Rs {:>9,.2f}".format(upper_fence))
print()
flagged = [a for a in amounts if a > upper_fence]
print("records above the fence:", len(flagged), "->", flagged)

The fence catches exactly one record, and it is the one the room found by eye three cells ago. The largest ordinary order at Rs 17,400 sits well inside it.

**A fence is a flag, never a delete key.** It says "look at this record". Yesterday's language holds without change: an outlier is a finding to investigate before it is a row to delete. The fence only finds it faster, on a file too large to eyeball.

## Section 4: reading the shape off sorted values

You have no charting library until Week 2. You do not need one for this.

Sort the column. Stand on the median. Look both ways.

```
 min          median                                        max
  |              |                                           |
  +--------------+-------------------------------------------+
     distance down          distance up
```

If the two distances are similar, the shape is roughly even. If one is far longer, that side has a tail, and the tail is what pulls the mean.

In [ ]:
median_amount = statistics.median(amounts)
down = median_amount - min(amounts)
up = max(amounts) - median_amount

print("median:        Rs {:>9,.2f}".format(median_amount))
print("distance down: Rs {:>9,.2f}".format(down))
print("distance up:   Rs {:>9,.2f}".format(up))
print("the up side is {:.0f} times the down side".format(up / down))
print()
print("mean / median ratio: {:.2f}".format(mean_amount / median_amount))

The tell you can use in any interview and on any dataset, with no formula at all:

| What you see | What it means |
|---|---|
| Mean noticeably larger than median | Something large is pulling on the right |
| Mean noticeably smaller than median | Something small is pulling on the left |
| Mean and median close together | The shape is roughly even, either one describes it |

On this file the mean is 2.4 times the median. You knew there was a tail before you looked at a single record.

Run those two numbers first on any new column. It costs one line and it tells you which statistic you are allowed to quote.

### Milestone 2: what you can now answer

**Where this shows up in production.** In 1973 the statistician Frank Anscombe built four datasets that share nearly identical means, variances and correlation coefficients, and look nothing like each other when drawn. The set is still handed to new analysts for one reason: a summary statistic is a compression, and every compression throws something away. Two datasets can hand you the same numbers and describe two different worlds.

Until Week 2 gives you a plotting library, the sorted list and the mean-to-median ratio are your picture. They are cheap, and they catch most of what a chart would have shown you.

**Interview question this section just made answerable.**

> How would you check for skew without plotting anything?

Sort the values, take the median, and compare the distance from the median to the maximum against the distance from the median to the minimum. Then compare the mean against the median: a mean well above the median means a right tail. Say the second part even if they only asked for one, because it is one line of code on any column in any language.

## What this notebook settled

| Question | Answer |
|---|---|
| Typical amount, honest version | Median, Rs 7,500 on 47 records |
| Typical amount, misleading version | Mean, Rs 18,000, which describes 1 record out of 47 |
| Why they differ | One real order of Rs 480,000 |
| Does the whale get deleted | No. It is real. The statistic changes, the data does not. |
| Shape, with no chart | Mean is 2.4 times the median, so there is a right tail |

**Crux.** The mean was right and the description was wrong. On a money column, send the median and say that is what you sent.

Notebook 2 takes this one level down, to the segment, where a second thing starts lying: the denominator.